# 내모델 json

In [ ]:
# 필수 라이브러리 설치
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

# Hugging Face 토큰 로그인 (실행 후 발급받은 토큰을 입력하세요)
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch
import glob
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. JSON을 LLM이 이해하기 쉬운 텍스트로 파싱하는 함수
def get_latest_vision_context(folder_path):
    # 폴더 내의 모든 json 파일 찾기
    files = glob.glob(f"{folder_path}/*.json")
    if not files:
        return "현재 기록된 위치 데이터가 없습니다."

    # 수정 시간 또는 파일명 기준으로 가장 최신 파일 선택
    files.sort()
    latest_file = files[-1]

    with open(latest_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 시간 및 장소 정보 추출
    timestamp = data.get("timestamp", "알 수 없는 시간")[:16].replace("T", " ") # "2026-04-05 18:39" 형태로 포맷팅
    location = data.get("location_context", "알 수 없는 장소")

    # LLM에게 줄 컨텍스트 문자열 생성
    context_text = f"[마지막 관측 시간: {timestamp}, 장소: {location}]\n"

    # 물체 정보 추출
    for obj in data.get("objects", []):
        name = obj.get("name", "알 수 없는 물건")
        surface = obj.get("position", {}).get("surface", "위치 불명")
        nearby = ", ".join(obj.get("nearby_objects", []))

        # 모델이 이해하기 쉽게 문장형으로 구성
        context_text += f"- {name}은(는) {surface}에 있습니다. (주변 물건: {nearby})\n"

    return context_text

# 2. Gemma 2B 모델 & 토크나이저 로드 (최신 2B instruct 모델 권장)
model_id = "google/gemma-2-2b-it"

print("모델 다운로드 중... (약 1~2분 소요)")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
print("모델 로드 완료!")

모델 다운로드 중... (약 1~2분 소요)


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

모델 로드 완료!


In [ ]:
import json

# 1. 특정 JSON 파일을 읽어서 텍스트로 변환하는 헬퍼 함수
def load_specific_json(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # JSON 파싱 (시간, 장소, 물건 위치 추출)
        timestamp = data.get("timestamp", "알 수 없는 시간")[:16].replace("T", " ")
        location = data.get("location_context", "알 수 없는 장소")

        context_text = f"[관측 시간: {timestamp}, 장소: {location}]\n"

        for obj in data.get("objects", []):
            name = obj.get("name", "알 수 없는 물건")
            surface = obj.get("position", {}).get("surface", "위치 불명")
            nearby = ", ".join(obj.get("nearby_objects", []))

            context_text += f"- {name}은(는) {surface}에 있습니다. (주변 물건: {nearby})\n"

        return context_text

    except FileNotFoundError:
        return "지정된 JSON 파일을 찾을 수 없습니다."
    except Exception as e:
        return f"파일을 읽는 중 오류가 발생했습니다: {e}"

# 2. 챗봇 함수 (특정 JSON 파일 경로를 입력받도록 수정)
def ask_vision_chatbot(user_question, target_json_path):
    # 1. 지정된 단일 JSON 파일에서 기록 가져오기
    context = load_specific_json(target_json_path)

    # 2. 시스템 프롬프트 구성
    prompt = f"""당신은 사용자의 물건 위치를 찾아주는 똑똑한 비서입니다.
아래 제공된 [카메라 기록]을 바탕으로 사용자의 질문에 대답하세요.
물건이 목록에 없으면 "현재 카메라 기록에 해당 물건이 없습니다."라고 정확히 말하세요.
주변 물건(nearby_objects) 정보도 함께 알려주면 사용자가 찾기 더 쉽습니다.

[카메라 기록]
{context}

[사용자 질문]
{user_question}"""

    # 3. 모델에 입력할 형태 세팅
    messages = [
        {"role": "user", "content": prompt}
    ]

    encoded_input = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # 4. 답변 생성
    outputs = model.generate(
        **encoded_input, # BatchEncoding 객체를 언패킹하여 전달
        max_new_tokens=150,
        temperature=0.1,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0][encoded_input['input_ids'].shape[-1]:], skip_special_tokens=True)
    return response.strip()

# --- 테스트 실행 ---
# 읽어올 특정 JSON 파일의 경로를 지정합니다. (예: 코랩에 업로드한 파일 경로)
my_json_file = "/content/내모델2.json"

print("질문: 내 노트북 어딨지?")
print("답변:", ask_vision_chatbot("내 노트북 어딨지?", my_json_file))
print("-" * 50)

print("질문: 불빛이나는 무언가 어디 뒀는지 알아?")
print("답변:", ask_vision_chatbot("불빛이나는 무언가 어디 뒀는지 알아?", my_json_file))

질문: 내 노트북 어딨지?
답변: 맥북은 테이블 위 중앙에 있습니다. (주변 물건: 아이폰, AirPods 케이스, MagSafe 충전기)
--------------------------------------------------
질문: 불빛이나는 무언가 어디 뒀는지 알아?
답변: 현재 카메라 기록에 불빛이나는 물건이 없습니다.


In [ ]:
import json

# 1. 새로운 JSON 구조를 파싱하는 헬퍼 함수
def load_specific_json(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # 'metadata' 부분 추출
        metadata = data.get("metadata", {})

        # 시간과 장소 (값이 null일 수 있으므로 대체 문자열 설정)
        captured_at = data.get("capturedAt")
        captured_at_str = captured_at if captured_at else "알 수 없는 시간"

        location = metadata.get("location")
        location_str = location if location else "알 수 없는 장소"

        # 씬 요약, 위치 힌트, 발견된 물건들 추출
        scene_summary = metadata.get("sceneSummary", "요약 정보 없음")
        position_hint = metadata.get("positionHint", "위치 힌트 없음")

        detected_objects = metadata.get("detectedObjects", [])
        objects_str = ", ".join(detected_objects) if detected_objects else "발견된 물건 없음"

        # LLM이 읽기 쉬운 형태로 컨텍스트 조립
        context_text = f"[관측 시간: {captured_at_str}, 장소: {location_str}]\n"
        context_text += f"- 전체 상황: {scene_summary}\n"
        context_text += f"- 물건들이 있는 대략적인 위치: {position_hint}\n"
        context_text += f"- 발견된 물건들: {objects_str}\n"

        return context_text

    except FileNotFoundError:
        return "지정된 JSON 파일을 찾을 수 없습니다."
    except Exception as e:
        return f"파일을 읽는 중 오류가 발생했습니다: {e}"

# 2. 챗봇 함수 (프롬프트 수정)
def ask_vision_chatbot(user_question, target_json_path):
    # 1. 지정된 단일 JSON 파일에서 기록 가져오기
    context = load_specific_json(target_json_path)

    # 2. 시스템 프롬프트 구성 (새로운 컨텍스트 구조에 맞게 지시사항 변경)
    prompt = f"""당신은 사용자의 물건 위치를 찾아주는 똑똑한 비서입니다.
아래 제공된 [카메라 기록]은 카메라가 포착한 전체 상황과 발견된 물건들의 목록입니다.
질문받은 물건이 '발견된 물건들' 목록에 없으면 "현재 카메라 기록에 해당 물건이 없습니다."라고 정확히 말하세요.
물건이 목록에 있다면, '전체 상황'과 '대략적인 위치'를 바탕으로 물건이 어디 있는지 유추해서 자연스럽게 대답해주세요.

[카메라 기록]
{context}

[사용자 질문]
{user_question}"""

    # 3. 모델에 입력할 형태 세팅
    messages = [
        {"role": "user", "content": prompt}
    ]

    # (주의: 사용하시는 토크나이저 설정에 따라 return_dict=True 가 필요할 수도 있습니다)
    encoded_input = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    # 4. 답변 생성
    outputs = model.generate(
        **encoded_input, # BatchEncoding 객체를 언패킹하여 전달
        max_new_tokens=150,
        temperature=0.1,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0][encoded_input['input_ids'].shape[-1]:], skip_special_tokens=True)
    return response.strip()

# --- 테스트 실행 ---
# 읽어올 특정 JSON 파일의 경로를 지정합니다.
my_json_file = "/content/관모델2.json"

print("질문: 내 노트북 어딨지?")
print("답변:", ask_vision_chatbot("내 노트북 어딨지?", my_json_file))
print("-" * 50)

print("질문: 에어팟 케이스 어디 뒀는지 알아?")
print("답변:", ask_vision_chatbot("에어팟 케이스 어디 뒀는지 알아?", my_json_file))

질문: 내 노트북 어딨지?
답변: 노트북은 작업 테이블 위에 있습니다. 맥북은 전자제품과 커피가 놓여 있는 테이블 위에 있습니다.
--------------------------------------------------
질문: 에어팟 케이스 어디 뒀는지 알아?
답변: 에어팟 케이스는 작업 테이블 위에 있습니다. 맥북과 커피가 놓여 있고, 램프, 컵, 아트 작품, 케이블도 함께 있습니다.


In [ ]:
import json
import glob
import os

# 1. 최신순으로 거슬러 올라가며 물건들의 '최종 위치'를 추적하는 함수
def get_all_latest_positions(folder_path):
    # 폴더 내 모든 json 파일 찾기 (예: /content/*.json)
    json_files = glob.glob(os.path.join(folder_path, "*.json"))

    if not json_files:
        return "폴더에 JSON 파일이 없습니다."

    all_data = []

    # 1단계: 모든 파일 읽어오기
    for file in json_files:
        try:
            with open(file, "r", encoding="utf-8") as f:
                data = json.load(f)
                all_data.append(data)
        except Exception as e:
            print(f"파일 읽기 실패 ({file}): {e}")

    # 2단계: JSON 내부의 timestamp를 기준으로 '최신순(내림차순)' 정렬
    # (시간 정보가 없으면 맨 뒤로 보냄)
    all_data.sort(key=lambda x: x.get("timestamp", ""), reverse=True)

    # 3단계: 최신 데이터부터 보면서 물건들의 위치 기록하기
    seen_objects = set() # 이미 찾은 물건 이름을 기록할 집합
    context_text = "[종합 관측 기록 (최신순 탐색 결과)]\n"

    for data in all_data:
        # 시간 포맷팅 (예: 2026-04-05 18:39)
        timestamp = data.get("timestamp", "시간 모름")[:16].replace("T", " ")
        location = data.get("location_context", "알 수 없는 장소")

        for obj in data.get("objects", []):
            name = obj.get("name", "알 수 없는 물건")

            # 핵심 로직: 아직 기록되지 않은 물건(즉, 가장 최신 기록)일 때만 추가
            if name not in seen_objects:
                surface = obj.get("position", {}).get("surface", "위치 불명")
                nearby = ", ".join(obj.get("nearby_objects", []))

                context_text += f"- {name}: {surface}에 있음 (마지막 관측: {timestamp}, {location} / 주변: {nearby})\n"

                # 물건을 찾았으니 목록에 추가 (이후 과거 데이터의 동일한 물건은 무시됨)
                seen_objects.add(name)

    if not seen_objects:
        return "관측된 물건 정보가 없습니다."

    return context_text

# 2. 똑똑해진 챗봇 함수
def ask_smart_vision_chatbot(user_question, target_folder):
    # 1. 최신순 탐색으로 완성된 종합 컨텍스트 가져오기
    context = get_all_latest_positions(target_folder)

    # 2. 시스템 프롬프트 구성
    prompt = f"""당신은 사용자의 물건 위치를 찾아주는 똑똑한 비서입니다.
아래 제공된 [종합 관측 기록]을 바탕으로 질문에 대답하세요. 이 기록은 각 물건이 '가장 마지막으로 목격된 시간과 위치'를 나타냅니다.
목록에 없는 물건을 찾으면 "해당 물건은 최근 기록에서 찾을 수 없습니다."라고 대답하세요.
관측 시간과 주변 물건 정보를 함께 알려주면 매우 좋습니다.

[종합 관측 기록]
{context}

[사용자 질문]
{user_question}"""

    # 3. 모델 실행 세팅
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # 4. 답변 생성
    outputs = model.generate(
        **input_ids, # BatchEncoding 객체를 언패킹하여 전달
        max_new_tokens=150,
        temperature=0.1,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0][input_ids['input_ids'].shape[-1]:], skip_special_tokens=True)
    return response.strip()

# --- 테스트 실행 ---
# 폴더 경로만 지정해주면, 그 안의 내모델.json, 관모델.json 등을 싹 다 뒤져서 스스로 최신순으로 맞춥니다!
target_dir = "/content"

print("질문: 내 컵이 어딨지?")
print("답변:", ask_smart_vision_chatbot("내 컵이 어딨지?", target_dir))

질문: 내 컵이 어딨지?
답변: 컵은 테이블 위에 있습니다. 마지막 관측은 2026-04-05 18:39이며 주변에 램프, 휴대전화, 케이블, 맥북이 있습니다.
